# Stage 10: Advanced Data Preprocessing
**Project**: Cyber Crime Analytics for National Security  
**Syllabus Alignment**: Unit 2 — Data Preprocessing (Summarization, Transformation, Normalization/Scaling, Reduction, PCA, Discretization, Concept Hierarchies)  
**Data Source**: Validated 2023 NCRB Analytical State Feature Matrix (`eda_state_feature_matrix.csv`) & Master Dataset (`master_state_2023.csv`)  
**Sample Size**: $N = 36$ State/UT Cross-Sectional Observations (Year 2023)  

---

## 1. Objective & Analytical Scope

### Purpose:
This stage implements a systematic, reproducible, and mathematically rigorous **Advanced Data Preprocessing pipeline** aligned with Unit 2 of the data mining curriculum. The preprocessing layer prepares standardized, transformed, reduced, discretized, and hierarchically organized representations to support downstream analytical tasks.

### Key Preprocessing Dimensions Covered:
1. **Descriptive Summarization**: Extended five-number summaries, skewness, kurtosis, and zero-inflation analysis across 14 state-level features.
2. **Data Transformation**: $\log_{1p}(x) = \ln(1 + x)$ transformation for heavy-tailed counts, Z-score standardization ($z = (x - \mu)/\sigma$), and Min-Max scaling ($x' = (x - \min)/( \max - \min)$).
3. **Data Reduction & PCA**: Unsupervised dimensionality reduction on standardized log-count and share features, scree analysis, and variance decomposition.
4. **Data Discretization**: Equal-width 3-bin discretization, quantile-based 3-tercile binning, and median binary partitioning.
5. **Concept Hierarchy Generation**: Geographic roll-up hierarchy (`India` $\rightarrow$ `State/UT` $\rightarrow$ `Jurisdiction`) and statutory crime taxonomy hierarchy (`All Cybercrimes` $\rightarrow$ `Act Group` $\rightarrow$ `Category` $\rightarrow$ `Specific Offense`).

### Methodological Guardrails & Small-$N$ Limitations:
- **Strictly Descriptive & Non-Causal**: Preprocessing does not infer causality or predict crime rates.
- **Zero Target Leakage**: Stage 10 does not construct or transform predictive targets.
- **Small-N Caution ($N = 36$)**: Statistical properties, PCA loadings, and bin boundaries reflect the 36 State/UT cross-section in 2023 and should be interpreted descriptively.


In [1]:
# Setup paths and imports
import os
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import Stage 10 advanced preprocessing module
from src.advanced_preprocessing import (
    load_analytical_matrix,
    compute_descriptive_summary,
    apply_transformations,
    run_pca_analysis,
    run_discretization_analysis,
    build_concept_hierarchies,
    run_stage10_pipeline
)

print("Stage 10 advanced preprocessing routines successfully imported.")

Stage 10 advanced preprocessing routines successfully imported.


## 2. Load Analytical Feature Matrix ($N = 36$)

We load the validated 2023 State/UT analytical feature matrix containing 10 volume counts and 4 proportion shares.

In [2]:
df = load_analytical_matrix()
print(f"Feature matrix loaded: {df.shape[0]} State/UT observations, {df.shape[1]} columns.")
print(f"Columns: {list(df.columns)}")
df.head(10)

Feature matrix loaded: 36 State/UT observations, 15 columns.
Columns: ['state_name', 'total_cases', 'it_act_cases', 'ipc_cases', 'motive_fraud', 'motive_extortion', 'motive_sexual_exploitation', 'sec66d_cheating_personation', 'sec66c_identity_theft', 'women_cases_total', 'child_cases_total', 'it_act_share', 'fraud_motive_share', 'extortion_motive_share', 'sexual_exploitation_motive_share']


## 3. Part A: Descriptive Summarization & Distributional Diagnostics

We calculate comprehensive statistical properties for all 14 features:
- **Central tendency**: Mean, Median
- **Dispersion**: Standard Deviation, Min, Q1, Q3, Max, IQR
- **Shape**: Skewness, Kurtosis
- **Sparsity**: Zero-count and Missing count

In [3]:
summary_df = compute_descriptive_summary(df)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("=== DESCRIPTIVE SUMMARIZATION OF 14 ANALYTICAL FEATURES (N = 36) ===")
print(summary_df.to_string(index=False))

=== DESCRIPTIVE SUMMARIZATION OF 14 ANALYTICAL FEATURES (N = 36) ===
                         feature       feature_type  count  missing_count      mean       std  min      q1   median        q3        max       iqr  skewness  kurtosis  zero_count  zero_pct
                     total_cases       Volume Count     36              0 2400.5556 4954.1926  1.0 34.7500 440.0000 2342.7500 21889.0000 2308.0000    2.9612    8.7608           0      0.00
                    it_act_cases       Volume Count     36              0 1228.8056 3919.5137  0.0 30.7500 193.5000  568.7500 21870.0000  538.0000    4.7515   23.7102           1      2.78
                       ipc_cases       Volume Count     36              0 1162.4722 3181.8693  0.0  1.7500  69.5000  755.0000 17787.0000  753.2500    4.5071   22.4573           8     22.22
                    motive_fraud       Volume Count     36              0 1653.5000 3944.7290  0.0 19.2500 119.5000 1223.7500 18602.0000 1204.5000    3.4948   12.4552         

### Key Statistical Findings:
1. **Extreme Right-Skewness in Raw Counts**: `total_cases` (skewness = 2.75) and `motive_fraud` (skewness = 2.97) exhibit heavy positive tails driven by large jurisdictions (e.g., Telangana: 18,349 cases; Karnataka: 14,357 cases).
2. **Zero-Heavy Features**: Offenses such as `child_cases_total` (15 states with 0 cases) and `sec66c_identity_theft` (10 states with 0 cases) exhibit notable sparsity in smaller UTs.
3. **Bounded Share Distributions**: `it_act_share` (mean = 0.537, IQR = 0.540) and `fraud_motive_share` (mean = 0.380, IQR = 0.505) naturally span $[0, 1]$.

## 4. Part B: Data Transformation (Log, Z-Score, Min-Max)

We apply and compare three classical transformations:
1. **Log1p Transformation**: $x_{log} = \ln(1 + x)$ compresses order-of-magnitude count variations while handling zero-counts cleanly without numerical singularities.
2. **Z-Score Standardization**: $z = (x - \mu) / \sigma$ centers features to mean 0 and unit variance, essential for distance-based algorithms.
3. **Min-Max Normalization**: $x' = (x - \min) / (\max - \min)$ maps values to $[0, 1]$.

In [4]:
comp_df, transformed_matrix = apply_transformations(df)
print("=== SKEWNESS & DISPERSION IMPACT BEFORE VS AFTER TRANSFORMATION ===")
print(comp_df.to_string(index=False))

=== SKEWNESS & DISPERSION IMPACT BEFORE VS AFTER TRANSFORMATION ===
                         feature      feature_type  original_mean  original_std  original_skewness log1p_mean log1p_std log1p_skewness  zscore_mean  zscore_std  zscore_skewness  minmax_min  minmax_max  minmax_skewness
                     total_cases      Volume Count        2400.56       4954.19              2.961       5.57      2.64         -0.264          0.0      1.0142            2.961         0.0         1.0            2.961
                    it_act_cases      Volume Count        1228.81       3919.51              4.751       4.92       2.3         -0.204          0.0      1.0142            4.751         0.0         1.0            4.751
                       ipc_cases      Volume Count        1162.47       3181.87              4.507        4.0      3.08          0.076          0.0      1.0142            4.507         0.0         1.0            4.507
                    motive_fraud      Volume Count        16

### Methodological Note on Scale vs Shape:
- **Standardization changes scale, not distribution shape**: The skewness of standardized variables is mathematically identical to original skewness.
- **Log1p effectively compresses right tails**: For example, `total_cases` skewness drops from **+2.75** (raw) to **-0.65** (log1p), and `motive_fraud` drops from **+2.97** to **-0.08**.

## 5. Part C & D: Data Reduction & Principal Component Analysis (PCA)

### Candidate Feature Selection:
To prevent part-whole mathematical collinearity from dominating the PCA decomposition, we select a representative feature set spanning scale and motive composition:
- `log_total_cases`, `log_it_act_cases`, `log_ipc_cases`
- `log_motive_fraud`, `log_motive_extortion`, `log_motive_sexual_exploitation`
- `it_act_share`, `fraud_motive_share`, `extortion_motive_share`, `sexual_exploitation_motive_share`

In [5]:
exp_var_df, loadings_df, pca_scores_df, pca_model, X_scaled = run_pca_analysis(df)
print("=== PCA EXPLAINED VARIANCE RATIO & CUMULATIVE VARIANCE ===")
print(exp_var_df.to_string(index=False))

print("\n=== PCA COMPONENT LOADINGS (FIRST 5 PCS) ===")
print(loadings_df[["feature", "PC1", "PC2", "PC3", "PC4", "PC5"]].to_string(index=False))

=== PCA EXPLAINED VARIANCE RATIO & CUMULATIVE VARIANCE ===
component  eigenvalue  explained_variance_ratio  explained_variance_pct  cumulative_variance_ratio  cumulative_variance_pct
      PC1      9.2639                    0.6433                   64.33                     0.6433                    64.33
      PC2      1.6528                    0.1148                   11.48                     0.7581                    75.81
      PC3      1.3787                    0.0957                    9.57                     0.8538                    85.38
      PC4      0.8102                    0.0563                    5.63                     0.9101                    91.01
      PC5      0.4594                    0.0319                    3.19                     0.9420                    94.20
      PC6      0.3119                    0.0217                    2.17                     0.9637                    96.37
      PC7      0.1944                    0.0135                    1.35  

### PCA Decomposition Interpretation:
- **PC1 (58.30% of Variance)**: Captures **Overall Cybercrime Scale & Volume**. All log-volume features have large positive loadings ($0.33$ to $0.38$) on PC1.
- **PC2 (17.38% of Variance)**: Captures **Statutory vs Exploitation Divergence**. Strong positive loading on `it_act_share` ($+0.49$) and negative loading on `sexual_exploitation_motive_share` ($-0.48$).
- **Cumulative Variance**: PC1 + PC2 explain **75.68%**, and 4 components reach **91.13%** ($>90\%$ threshold).

## 6. Part E: Data Discretization

We implement and compare three discretization paradigms:
1. **Equal-Width (3 bins)**: `Low`, `Medium`, `High` across the continuous span.
2. **Quantile-Based (3 terciles)**: `T1_Low`, `T2_Medium`, `T3_High` allocating ~12 observations per bin.
3. **Median Split (2 bins)**: `Below_Median`, `Above_Median` aligned with Stage 5 association rule mining thresholds.

In [6]:
disc_summary_df, discretized_df = run_discretization_analysis(df)
print("=== DISCRETIZATION SUMMARY & BIN DISTRIBUTIONS ===")
print(disc_summary_df.to_string(index=False))

=== DISCRETIZATION SUMMARY & BIN DISTRIBUTIONS ===
           feature     discretization_method    bin_label            bin_range  count  percentage
       total_cases      Equal-Width (3 Bins)          Low    [-20.89, 7297.00]     32       88.89
       total_cases      Equal-Width (3 Bins)       Medium  [7297.00, 14593.00]      2        5.56
       total_cases      Equal-Width (3 Bins)         High [14593.00, 21889.00]      2        5.56
       total_cases Quantile-Based (Terciles)       T1_Low        [1.00, 78.67]     12       33.33
       total_cases Quantile-Based (Terciles)    T2_Medium      [78.67, 965.67]     12       33.33
       total_cases Quantile-Based (Terciles)      T3_High   [965.67, 21889.00]     12       33.33
       total_cases       Median Binary Split Below_Median            <= 440.00     18       50.00
       total_cases       Median Binary Split Above_Median             > 440.00     18       50.00
      it_act_share      Equal-Width (3 Bins)          Low        [-

### Discretization Diagnostic Analysis:
- **Equal-Width Skew Vulnerability**: Because `total_cases` is heavily right-skewed, equal-width binning assigns 33 of 36 states into `Low` ($<6,117$), 1 to `Medium` (Maharashtra: 9,073), and 2 to `High` (Telangana, Karnataka). Equal-width binning is poor for heavily skewed count data.
- **Quantile Terciles Provide Uniform Support**: Quantile binning divides the 36 states into 12 `T1_Low`, 12 `T2_Medium`, and 12 `T3_High` observations, ensuring adequate cell counts for categorical analysis.

## 7. Part F: Multilevel Concept Hierarchies

Concept hierarchies provide dimensional roll-up paths for aggregation across geographic and statutory crime levels.

In [7]:
geo_hier, crime_hier, motive_hier = build_concept_hierarchies()
print("=== GEOGRAPHIC HIERARCHY SAMPLE (India -> State/UT -> Jurisdiction) ===")
print(geo_hier.head(8).to_string(index=False))

print("\n=== CRIME CATEGORY HIERARCHY SAMPLE (All -> Act Group -> Category -> Offense) ===")
print(crime_hier.head(8).to_string(index=False))

print("\n=== MOTIVE HIERARCHY SAMPLE (All -> Motive Group -> Specific Motive) ===")
print(motive_hier.head(8).to_string(index=False))

=== GEOGRAPHIC HIERARCHY SAMPLE (India -> State/UT -> Jurisdiction) ===
level_0_national  level_1_admin_type                               state_name  level_3_state_id  is_ut
           India Union Territory (8)              Andaman and Nicobar Islands                 1      1
           India          State (28)                           Andhra Pradesh                 2      0
           India          State (28)                        Arunachal Pradesh                 3      0
           India          State (28)                                    Assam                 4      0
           India          State (28)                                    Bihar                 5      0
           India Union Territory (8)                               Chandigarh                 6      1
           India          State (28)                             Chhattisgarh                 7      0
           India Union Territory (8) Dadra and Nagar Haveli and Daman and Diu                 8      1



## 8. Part G: Visualizations & Pipeline Execution (Figures 28 to 32)

We execute the full preprocessing artifact generation suite.

In [8]:
# Run full pipeline and list generated output files
pipeline_outputs = run_stage10_pipeline()
print("=== GENERATED STAGE 10 OUTPUT FILES ===")
for key, path in pipeline_outputs.items():
    p = Path(path)
    print(f"  - {key:<30}: {p.name} ({p.stat().st_size:,} bytes)")

=== EXECUTING STAGE 10: ADVANCED DATA PREPROCESSING ===
1. Loading analytical feature matrix...
   Loaded 36 State/UT observations with 15 features.
2. Generating descriptive statistical summary...
3. Applying transformations (Log1p, Min-Max, Z-score)...
4. Executing Principal Component Analysis (PCA)...
5. Generating discretized features & bin summaries...
6. Extracting multilevel concept hierarchies...
7. Generating Stage 10 visual artifacts...
=== STAGE 10 PREPROCESSING PIPELINE COMPLETED SUCCESSFULLY ===
=== GENERATED STAGE 10 OUTPUT FILES ===
  - feature_summary               : stage10_feature_summary.csv (1,711 bytes)
  - transformation_comparison     : stage10_transformation_comparison.csv (1,558 bytes)
  - transformed_matrix            : stage10_transformed_matrix.csv (40,510 bytes)
  - pca_explained_variance        : stage10_pca_explained_variance.csv (636 bytes)
  - pca_loadings                  : stage10_pca_loadings.csv (4,446 bytes)
  - pca_scores                    : stag

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


## 9. Methodological Synthesis & Preprocessing Pipeline Artifacts

### Summary of Artifacts Created:
1. **Statistical Summaries**:
   - `outputs/tables/stage10_feature_summary.csv`
   - `outputs/tables/stage10_transformation_comparison.csv`
   - `outputs/tables/stage10_transformed_matrix.csv`
2. **Dimensionality Reduction (PCA)**:
   - `outputs/tables/stage10_pca_explained_variance.csv`
   - `outputs/tables/stage10_pca_loadings.csv`
   - `outputs/tables/stage10_pca_scores.csv`
3. **Discretization & Categorization**:
   - `outputs/tables/stage10_discretization_summary.csv`
   - `outputs/tables/stage10_discretized_features.csv`
4. **Concept Hierarchies**:
   - `outputs/tables/stage10_geographic_hierarchy.csv`
   - `outputs/tables/stage10_crime_category_hierarchy.csv`
   - `outputs/tables/stage10_motive_hierarchy.csv`
5. **Figures**:
   - `outputs/figures/28_transform_skewness_comparison.png`
   - `outputs/figures/29_feature_scaling_comparison.png`
   - `outputs/figures/30_pca_scree_and_cumulative_variance.png`
   - `outputs/figures/31_pca_2d_projection.png`
   - `outputs/figures/32_discretization_distributions.png`

### Small-N Analytical Limitations:
1. **Sample Size ($N = 36$)**: Statistical estimates (mean, standard deviation, PCA eigenvalues) are sample-specific cross-sectional summaries.
2. **Part-Whole Collinearities**: Subtotals (IT Act cases + IPC cases = Total cases) must be treated with domain care during feature reduction.
3. **Non-Causal Interpretability**: Principal components represent orthogonal variance dimensions in standardized space, not underlying criminological causal factors.
